# Pengujian Model Fraud dengan TensorFlow Serving

Notebook ini mengirim prediction request ke TensorFlow Serving pada `http://localhost:8501`. Dari root project, jalankan `docker compose -f deployment/docker-compose.serving.yml up -d` setelah pipeline menghasilkan SavedModel.

Model Trainer menerima fitur hasil transformasi dengan akhiran `_xf`, sehingga request di bawah menormalisasi fitur menggunakan statistik dataset sebelum dikirim.

In [30]:
import json
import base64
import requests
import tensorflow as tf
import pandas as pd
from pathlib import Path

# Konfigurasi URL dan Path Dataset
SERVING_URL = "http://localhost:8501/v1/models/fraud_detection:predict"

PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (candidate / "data" / "creditcard.csv").is_file() and (candidate / "serving_model").is_dir():
        PROJECT_ROOT = candidate
        break

DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"
df = pd.read_csv(DATA_PATH)

# Siapkan Payload dengan format tf.train.Example berenkode base64
raw_sample = df.iloc[0]
feature_dict = {}

# Fitur V1 - V28
for i in range(1, 29):
    feat_key = f'"V{i}"'
    val = float(raw_sample[f"V{i}"])
    feature_dict[feat_key] = tf.train.Feature(float_list=tf.train.FloatList(value=[val]))

# Fitur Amount dan Time
for feat_key, orig_col in [('"Amount"', 'Amount'), ('"Time"', 'Time')]:
    val = float(raw_sample[orig_col])
    feature_dict[feat_key] = tf.train.Feature(float_list=tf.train.FloatList(value=[val]))

# Tambahkan fitur Class dengan tipe int64_list sesuai ekspektasi skema
feature_dict['"Class"'] = tf.train.Feature(int64_list=tf.train.Int64List(value=[0]))

example = tf.train.Example(features=tf.train.Features(feature=feature_dict))
serialized_example = example.SerializeToString()
encoded_example = base64.b64encode(serialized_example).decode('utf-8')

payload = {
    "instances": [
        {
            "examples": {"b64": encoded_example}
        }
    ]
}

# Kirim Request ke TensorFlow Serving
try:
    response = requests.post(SERVING_URL, json=payload, timeout=30)
except requests.exceptions.ConnectionError:
    print(
        "TensorFlow Serving belum aktif di port 8501. "
        "Buka Docker Desktop, lalu jalankan: "
        "docker compose -f deployment/docker-compose.serving.yml up -d"
    )
else:
    print(f"HTTP status: {response.status_code}")
    if response.status_code != 200:
        print("Detail Error dari Server:", response.text)
    else:
        response.raise_for_status()
        prediction = response.json()
        print("Hasil Prediksi:", json.dumps(prediction, indent=2))

HTTP status: 200
Hasil Prediksi: {
  "predictions": [
    [
      0.511104524
    ]
  ]
}


## Pemeriksaan hasil

Nilai probabilitas berada pada `predictions[0][0]`. Request ini melewati TensorFlow Serving REST API, bukan endpoint Flask. Jika mendapat connection error, pastikan container aktif dan model version sudah tersedia di `serving_model/fraud_detection`.